# OneVoice — Synthetic Noisy Construction Speech Generator
Sequential TTS (rate-limit safe) · Auto-Resume Checkpoint · Noise Auto-Repair · Python 3.12

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — Mount Drive & Install
# ═══════════════════════════════════════════════════════════
import os
IN_COLAB = 'google.colab' in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = '/content/drive/MyDrive/onevoice_audio_v1'
else:
    OUTPUT_ROOT = '/kaggle/working/onevoice_audio_v1'
print('Output:', OUTPUT_ROOT)
!pip install -q edge-tts soundfile librosa pandas tqdm nest_asyncio

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — Clone Dataset
# ═══════════════════════════════════════════════════════════
if not os.path.exists('/content/OneVoice'):
    !git clone --depth 1 https://github.com/Platypus27-coder/OneVoice.git /content/OneVoice
else:
    print('Repo already cloned.')

for p in ['data/onevoice_construction_v2', 'onevoice-edge/data/onevoice_construction_v2']:
    full = os.path.join('/content/OneVoice', p)
    if os.path.exists(full):
        DATA_DIR = full
        break
else:
    raise FileNotFoundError('Cannot find onevoice_construction_v2!')

print('Data:', DATA_DIR)
print('Files:', os.listdir(DATA_DIR))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — Config
# ═══════════════════════════════════════════════════════════
import random, json, re, time

NOISE_DIR   = os.path.join(OUTPUT_ROOT, 'noise_bank')
CLEAN_DIR   = os.path.join(OUTPUT_ROOT, 'clean')
NOISY_DIR   = os.path.join(OUTPUT_ROOT, 'noisy')
MANIFEST    = os.path.join(OUTPUT_ROOT, 'manifest.jsonl')
for d in [OUTPUT_ROOT, NOISE_DIR, CLEAN_DIR, NOISY_DIR]:
    os.makedirs(d, exist_ok=True)

SAMPLES_PER_TEXT = 2
SAMPLE_RATE      = 16000
MAX_UTTERANCES   = None   # None = all 8064

VI_SPEAKERS = ['vi-VN-HoaiMyNeural', 'vi-VN-NamMinhNeural']
NOISE_CLASSES = [
    'excavator.wav','angle_grinder.wav','drilling.wav','hammer.wav',
    'diesel_engine.wav','generator.wav','truck.wav','wind.wav','worker_babble.wav',
]
SNR_OPTIONS = [0, 5, 10, 15, 20]
print('Config OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — Download & Auto-Repair Noise Bank
# ═══════════════════════════════════════════════════════════
import numpy as np, soundfile as sf

ESC50 = 'https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio'
NOISE_URLS = {
    'excavator.wav':     f'{ESC50}/1-116765-A-41.wav',
    'angle_grinder.wav': f'{ESC50}/3-156897-A-13.wav',
    'drilling.wav':      f'{ESC50}/4-182368-A-12.wav',
    'hammer.wav':        f'{ESC50}/3-149189-A-13.wav',
    'diesel_engine.wav': f'{ESC50}/1-26143-A-43.wav',
    'generator.wav':     f'{ESC50}/2-109371-A-43.wav',
    'truck.wav':         f'{ESC50}/5-219213-A-11.wav',
    'wind.wav':          f'{ESC50}/1-179701-A-25.wav',
    'worker_babble.wav': f'{ESC50}/1-26143-A-43.wav',
}

def synth_noise(name, dur=10, sr=16000):
    t = np.linspace(0, dur, sr*dur)
    if 'grinder' in name or 'drill' in name:
        n = .6*np.sin(2*np.pi*3200*t+np.sin(2*np.pi*50*t))+.4*np.random.randn(len(t))
    elif any(k in name for k in ('engine','excavator','generator','truck')):
        n = .5*np.sin(2*np.pi*60*t)+.3*np.sin(2*np.pi*120*t)+.3*np.random.randn(len(t))
    elif 'hammer' in name:
        n = .2*np.random.randn(len(t))
        for i in np.arange(0,len(t),int(sr*.8),dtype=int):
            e=min(i+int(sr*.05),len(t)); n[i:e]+=np.random.randn(e-i)*3
    else:
        n = np.convolve(np.random.randn(len(t)),[.05,-.09,.05],mode='same')
    return np.clip(n/(np.max(np.abs(n))+1e-9),-1,1)

def ok_wav(p):
    try: return os.path.exists(p) and os.path.getsize(p)>10000 and sf.read(p) is not None
    except: return False

for fname,url in NOISE_URLS.items():
    dst=os.path.join(NOISE_DIR,fname)
    if ok_wav(dst): print(f'  OK {fname}'); continue
    if os.path.exists(dst): os.remove(dst)
    print(f'  DL {fname}...')
    os.system(f'wget -q -O "{dst}" "{url}"')
    if not ok_wav(dst):
        print(f'  -> synth {fname}')
        if os.path.exists(dst): os.remove(dst)
        sf.write(dst, synth_noise(fname), 16000)
print('Noise bank OK')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — TTS + Audio Functions
#   Sequential TTS: 1 request at a time = no rate-limit
#   Retry 3x with exponential backoff
# ═══════════════════════════════════════════════════════════
import numpy as np, soundfile as sf, librosa
import asyncio, edge_tts, nest_asyncio
nest_asyncio.apply()

def clean_text(text):
    text = re.sub(r'[^\w\s\u00C0-\u024F\u1E00-\u1EFF.,!?;:()\'\-]', ' ', str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text if len(text) > 2 else None

def tts_one(text, out_path, voice):
    """Synthesize ONE file. Retry 3x with backoff. Returns True/False."""
    for attempt in range(3):
        try:
            if attempt > 0: time.sleep(1.5 * attempt)
            loop = asyncio.get_event_loop()
            loop.run_until_complete(
                edge_tts.Communicate(text, voice).save(out_path)
            )
            return True
        except Exception as e:
            if attempt == 2:
                print(f'  [SKIP] {os.path.basename(out_path)}: {e}')
                return False
    return False

def apply_rir(speech, sr):
    delay = int(sr * random.uniform(0.03, 0.08))
    echo = np.zeros_like(speech)
    echo[delay:] = speech[:-delay] * random.uniform(0.2, 0.4)
    out = speech + echo
    return out / (np.max(np.abs(out)) + 1e-9)

def mix_noise(speech, noise, snr_db):
    if len(noise) < len(speech):
        noise = np.tile(noise, int(np.ceil(len(speech)/len(noise))))
    noise = noise[:len(speech)]
    rs = np.sqrt(np.mean(speech**2)+1e-9)
    rn = np.sqrt(np.mean(noise**2)+1e-9)
    m = speech + (rs/(rn*(10**(snr_db/20))))*noise
    return np.clip(m/(np.max(np.abs(m))+1e-9),-1,1)

def augment(a):
    a = a * (10**(random.uniform(-3,3)/20))
    if random.random()<.05:
        t=random.uniform(.7,.95); a=np.clip(a,-t,t)
    return a

print('Functions OK (sequential TTS, retry=3)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — GENERATE DATASET
#   - Auto-resume from checkpoint
#   - Each wav + manifest line flushed to Drive instantly
#   - Safe to interrupt and re-run anytime
# ═══════════════════════════════════════════════════════════
import pandas as pd
from tqdm.notebook import tqdm

# ── Load checkpoint ──────────────────────────────────────
existing = set()
if os.path.exists(MANIFEST):
    for line in open(MANIFEST, encoding='utf-8'):
        try: existing.add(json.loads(line.strip()).get('audio'))
        except: pass
    print(f'Checkpoint: {len(existing)} samples done')

# ── Load utterances ──────────────────────────────────────
df = pd.read_csv(os.path.join(DATA_DIR, 'utterances_all.csv'))
if MAX_UTTERANCES: df = df.head(MAX_UTTERANCES)
print(f'Utterances: {len(df)}')

# ── Load noise ───────────────────────────────────────────
NC = {}
for nc in NOISE_CLASSES:
    p = os.path.join(NOISE_DIR, nc)
    if not os.path.exists(p): continue
    try: NC[nc],_ = librosa.load(p, sr=SAMPLE_RATE, mono=True)
    except:
        s=synth_noise(nc); sf.write(p,s,SAMPLE_RATE); NC[nc]=s
if not NC: NC['_silence']=np.zeros(SAMPLE_RATE)
noises = list(NC.keys())
print(f'Noise types: {noises}')

# ── Main loop ────────────────────────────────────────────
total = len(existing)
skip  = 0

with open(MANIFEST, 'a', encoding='utf-8') as mf:
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Generating'):
        uid  = str(row['utterance_id'])
        vi   = str(row['vi'])
        en   = str(row.get('en',''))
        dom  = str(row.get('domain','unknown'))
        inte = str(row.get('intent','unknown'))
        risk = str(row.get('risk_level','unknown'))
        spl  = str(row.get('split','train'))

        # Skip if all variants exist
        if all(
            f'{uid}_n{v+1:02d}.wav' in existing
            for v in range(SAMPLES_PER_TEXT)
        ): continue

        # TTS clean wav (sequential, retry=3)
        cf = f'{uid}_clean.wav'
        cp = os.path.join(CLEAN_DIR, cf)
        if not os.path.exists(cp):
            txt = clean_text(vi)
            if not txt: skip+=1; continue
            voice = random.choice(VI_SPEAKERS)
            if not tts_one(txt, cp, voice):
                skip+=1; continue

        try: ca,_ = librosa.load(cp, sr=SAMPLE_RATE, mono=True)
        except: skip+=1; continue

        rev = apply_rir(ca, SAMPLE_RATE)
        voice = random.choice(VI_SPEAKERS)

        for v in range(SAMPLES_PER_TEXT):
            nf = f'{uid}_n{v+1:02d}.wav'
            np_ = os.path.join(NOISY_DIR, nf)
            if nf in existing: continue

            nn  = random.choice(noises)
            snr = random.choice(SNR_OPTIONS)
            rev_on = random.random() > 0.35
            mixed = augment(mix_noise(rev if rev_on else ca, NC[nn], snr))
            sf.write(np_, mixed, SAMPLE_RATE)

            mf.write(json.dumps({
                'audio':nf, 'clean_audio':cf,
                'text':vi, 'translation':en,
                'domain':dom, 'intent':inte,
                'risk_level':risk, 'split':spl,
                'speaker_id':voice,
                'noise_type':nn.replace('.wav',''),
                'snr_db':snr, 'reverb':rev_on,
                'rir_id':'simulated_echo' if rev_on else 'none',
                'synthetic_speech':True, 'synthetic_noise_mix':True,
                'sample_rate':SAMPLE_RATE,
            }, ensure_ascii=False)+'\n')
            mf.flush()
            existing.add(nf)
            total += 1

print(f'\nDone! Total: {total} | Skipped: {skip}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — Stats
# ═══════════════════════════════════════════════════════════
import pandas as pd
entries = [json.loads(l) for l in open(MANIFEST, encoding='utf-8') if l.strip()]
dm = pd.DataFrame(entries)
print(f'Total: {len(dm)}')
for c in ['domain','noise_type','snr_db','split']:
    print(f'\n{c}:'); print(dm[c].value_counts().to_string())

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — Listen Test
# ═══════════════════════════════════════════════════════════
from IPython.display import Audio, display
s = random.choice(entries)
print(f"Text: {s['text']}")
print(f"Noise: {s['noise_type']} | SNR: {s['snr_db']}dB")
print('Clean:'); display(Audio(os.path.join(CLEAN_DIR,s['clean_audio']),rate=SAMPLE_RATE))
print('Noisy:'); display(Audio(os.path.join(NOISY_DIR,s['audio']),rate=SAMPLE_RATE))